# Tamanho dos grupos de datasets dos Dados Abertos da CAPES

Este notebook apresenta uma análise quantitativa e estrutural dos conjuntos de dados disponibilizados pela Coordenação de Aperfeiçoamento de Pessoal de Nível Superior (CAPES) em seu portal de dados abertos.

## 📌 Contextualização

Visualmente, o portal da CAPES organiza seu catálogo em 4 grandes eixos temáticos (organizações):

    - Acessos ao Portal de Periódicos da CAPES

    - Avaliação da Pós-Graduação Stricto Sensu

    - Bolsas e auxílios concedidos

    - Informações orçamentárias

Nota: Ao consultar a API nativa (CKAN) que alimenta o portal, nota-se que categorias amplas como "Educação" e "Pesquisa e Desenvolvimento", são aplicadas de forma transversal a quase totalidade dos conjuntos de dados (datasets), servindo como grandes marcadores institucionais e não como chaves de particionamento.


## 🎯 Objetivo da Análise

O objetivo deste estudo é demonstrar, através de métricas de volume (peso em Megabytes) e contagem de datasets, que as bases de dados referentes à pós-graduação e avaliação (Stricto Sensu) — que englobam catálogos de teses, produções intelectuais e discentes — são significativamente mais numerosas e volumosas em comparação aos demais eixos da fundação, refletindo o foco da instituição e o potencial elevado destes conjuntos de dados para gestão acadêmica e análises cientométricas.


In [250]:
##Imports
import requests
import urllib3
import time
import pandas as pd
from collections import defaultdict
import json
import os

In [251]:
##Opções do pandas
pd.set_option('display.max_colwidth', None)
#pd.set_option('display.max_rows', None)

# Suprime os avisos de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#URL da API
URL = "https://dadosabertos.capes.gov.br/api/3/action/package_search"

#Diretorio de saída de dados
outdir = 'outputs'


os.makedirs(outdir, exist_ok=True)

#Caminho dos arquivos a serem salvos localmente
json_path = os.path.join(outdir, '01_datasets_capes_info.json')
csv_arquivos_allformats_path = os.path.join(outdir, '01_arquivos_allformats.csv')
csv_arquivos_onlycsv_path = os.path.join(outdir, '01_arquivos_onlycsv.csv')
csv_datasets_onlycsv_path = os.path.join(outdir, '01_datasets_onlycsv.csv')
csv_grupos_onlycsv_path = os.path.join(outdir, '01_grupos_onlycsv.csv')
csv_grupos_onlycsv_posgrad_path = os.path.join(outdir, '01_grupos_onlycsv_posgrad.csv')
csv_org_onlycsv_path = os.path.join(outdir, '01_organizacoes_onlycsv.csv')

In [252]:
#Recuperar todos os metadados dos datasets da CAPES

def coletar_todos_datasets():
    """
    Varre a API da CAPES e baixa os metadados de todos os datasets disponíveis.
    Retorna uma lista de dicionários, onde cada dicionário é um dataset completo.
    """
    todos_os_datasets = []
    
    start = 0
    rows = 1000  # Pede o limite máximo do CKAN por vez para ser mais rápido
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    print("Iniciando o download de todos os metadados da CAPES...")
    
    while True:
        params = {
            'q': '*:*',
            'rows': rows,
            'start': start
        }
        
        print(f"Baixando lote: {start} a {start + rows}...")
        
        sucesso = False
        for tentativa in range(3):
            try:
                response = requests.get(URL, params=params, headers=headers, verify=False, timeout=60)
                response.raise_for_status()
                sucesso = True
                break
            except requests.exceptions.Timeout:
                print(f"  -> Timeout na tentativa {tentativa + 1}. Aguardando 5 segundos...")
                time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"  -> Erro de conexão: {e}. Aguardando...")
                time.sleep(5)
                
        if not sucesso:
            raise Exception("O servidor não respondeu após 3 tentativas. Abortando.")

        dados = response.json()
        datasets_da_pagina = dados.get('result', {}).get('results', [])
        
        # Se a página vier vazia, chegamos ao fim dos dados
        if not datasets_da_pagina:
            print(f"\nDownload concluído! Total de datasets armazenados na memória: {len(todos_os_datasets)}")
            break
            
        # Adiciona os datasets desta página na nossa lista mestre
        todos_os_datasets.extend(datasets_da_pagina)
        
        start += rows 
        
        # Uma pequena pausa de 1 segundo para ser educado com o servidor e evitar bloqueios (HTTP 429)
        time.sleep(1) 
        
    return todos_os_datasets

In [253]:
#Baixando todos os metadados 
metadados = coletar_todos_datasets()


#Salvando metadados em arquivo json local
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=4)
print("Metadados CAPES salvos com sucesso!")

Iniciando o download de todos os metadados da CAPES...
Baixando lote: 0 a 1000...
Baixando lote: 1000 a 2000...

Download concluído! Total de datasets armazenados na memória: 93
Metadados CAPES salvos com sucesso!


In [254]:
#Carregando metadados de arquivo json local (precisa ter rodado a célula acima pelo menos uma vez)
with open(json_path, "r", encoding="utf-8") as f:
    metadados = json.load(f)

In [255]:
#Analisando os primeiros registros
metadados[:2]

[{'author': '',
  'author_email': '',
  'creator_user_id': '7fbacb25-45fb-4fbe-b0af-d096ae71912f',
  'id': 'f4684855-ec35-49c3-8071-eea290fbd88b',
  'isopen': True,
  'license_id': 'cc-by',
  'license_title': 'Creative Commons Atribuição',
  'license_url': 'http://www.opendefinition.org/licenses/cc-by',
  'maintainer': 'DEB',
  'maintainer_email': 'OUVIDORIA@CAPES.GOV.BR',
  'metadata_created': '2025-06-25T12:05:27.941514',
  'metadata_modified': '2026-06-30T14:27:02.329688',
  'name': '2013-a-2025-participantes-de-programas-de-formacao-de-professores-da-educacao-basica-no-exterior',
  'notes': 'Os dados disponibilizados envolvem os Participantes de Programas de Formação de Professores da Educação Básica no Exterior, no período de 2013 a 2025.\r\n\r\nEsta publicação abrange os dados dos anos de 2013 a 2025, conforme o cronograma estabelecido no PDA 2025-2027.',
  'num_resources': 6,
  'num_tags': 3,
  'organization': {'id': '11b0c6b9-fe2e-4cb1-89a1-6e481a1c7b29',
   'name': 'bolsas-e-a

In [256]:
#Gerar dataframe com os dados da CAPES

# Lista de grupos genéricos da CAPES que queremos ignorar no json
GRUPOS_GENERICOS = ['Educação', 'Pesquisa e Desenvolvimento']

def extrair_tabela_metadados(metadados_capes_json):
    linhas = []
    
    for dataset in metadados_capes_json:
        # 1. Extrai o ID/Título do Dataset (raiz do dicionário)
        dataset_id = dataset.get('id', 'Sem ID')
        dataset_titulo = dataset.get('title', 'Sem Título')
        
        # 2. Extrai o nome da organização
        org_nome = dataset.get('organization', {}).get('name', 'Sem Organização')
        
        # 3. Extrai os grupos (display_name)
        grupos_brutos = dataset.get('groups', [])
        
        # Extrai os nomes de todos os grupos atrelados ao dataset
        todos_os_grupos = [g.get('display_name') for g in grupos_brutos]
        
        # Remove os grupos genéricos da lista usando uma compreensão de lista
        grupos_especificos = [g for g in todos_os_grupos if g not in GRUPOS_GENERICOS]
        
        # Pega o primeiro grupo que restou (que será o tema real), ou usa um padrão
        grupo_principal = grupos_especificos[0] if grupos_especificos else 'Outros / Sem Eixo Específico'
        
        # 4. Iteração sobre os recursos (arquivos)
        recursos = dataset.get('resources', [])
        for recurso in recursos:
            # Extrai os dados específicos do arquivo/recurso
            arquivo_id = recurso.get('id', 'Sem ID')
            arquivo_nome = recurso.get('name', 'Sem nome de arquivo')
            formato = recurso.get('format', '').upper()
            
            # Tratamento e conversão de tamanho
            tamanho_bytes = recurso.get('size')
            tamanho_bytes = int(tamanho_bytes) if tamanho_bytes is not None else 0
            tamanho_mb = tamanho_bytes / (1024 * 1024)
            
            # Monta o dicionário na ordem exata solicitada
            linhas.append({
                'Arquivo (ID)': arquivo_id,
                'Arquivo (Nome)': arquivo_nome,
                'Dataset (ID)': dataset_id,
                'Dataset (Título)': dataset_titulo,
                'Grupo': grupo_principal,
                'Organização': org_nome,
                'Formato': formato,
                'Tamanho (Bytes)': tamanho_bytes,
                'Tamanho (MB)': round(tamanho_mb, 2)
            })
                
    # Cria e retorna o DataFrame final
    df = pd.DataFrame(linhas)
    return df


In [257]:
df_arquivos = extrair_tabela_metadados(metadados)

df_arquivos

,Arquivo (ID),Arquivo (Nome),Dataset (ID),Dataset (Título),Grupo,Organização,Formato,Tamanho (Bytes),Tamanho (MB)
0,48fe1036-db0d-40ce-a964-983407a5f105,PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2013A2024-2025-06-01,f4684855-ec35-49c3-8071-eea290fbd88b,[2013 a 2025] Participantes de Programas de Formação de Professores da Educação Básica no Exterior,Participantes de Programas de Formação de Professores da Educação Básica no Exterior,bolsas-e-auxilios,CSV,297924,0.28
1,592065bc-2f60-4647-b792-924b038986f0,PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2013A2024-2025-06-01,f4684855-ec35-49c3-8071-eea290fbd88b,[2013 a 2025] Participantes de Programas de Formação de Professores da Educação Básica no Exterior,Participantes de Programas de Formação de Professores da Educação Básica no Exterior,bolsas-e-auxilios,XLSX,91468,0.09
2,5452f35a-7ebb-4aae-8cc5-0f85df99d35d,PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2025-2026-06-01,f4684855-ec35-49c3-8071-eea290fbd88b,[2013 a 2025] Participantes de Programas de Formação de Professores da Educação Básica no Exterior,Participantes de Programas de Formação de Professores da Educação Básica no Exterior,bolsas-e-auxilios,CSV,178225,0.17
3,8867d36b-2cd8-4a05-97c4-31e637096e94,PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2025-2026-06-01,f4684855-ec35-49c3-8071-eea290fbd88b,[2013 a 2025] Participantes de Programas de Formação de Professores da Educação Básica no Exterior,Participantes de Programas de Formação de Professores da Educação Básica no Exterior,bolsas-e-auxilios,XLSX,49486,0.05
4,854a2c0e-d4cc-4767-bf41-684add6ae8fd,Metadados dos dados de Participantes de Programas de Formação de Professores da Educação Básica no Exterior,f4684855-ec35-49c3-8071-eea290fbd88b,[2013 a 2025] Participantes de Programas de Formação de Professores da Educação Básica no Exterior,Participantes de Programas de Formação de Professores da Educação Básica no Exterior,bolsas-e-auxilios,PDF,167298,0.16
...,...,...,...,...,...,...,...,...,...
1400,2dc8fed8-f830-439e-8cce-8d81af50ca63,BR-CAPES-COLSUCUP-DOCENTE-2016-2023-08-01,35eab2f8-5a64-4619-b3f1-63a2e6690cfa,[2013 a 2016] Docentes da Pós-Graduação Stricto Sensu no Brasil,Docentes da Pós-Graduação Stricto Sensu no Brasil,diretoria-de-avaliacao,XLSX,16477872,15.71
1401,b871db42-a86e-43d3-bf21-2ea2e85d0acd,Metadados dos dados de Docentes da Pós-Graduação,35eab2f8-5a64-4619-b3f1-63a2e6690cfa,[2013 a 2016] Docentes da Pós-Graduação Stricto Sensu no Brasil,Docentes da Pós-Graduação Stricto Sensu no Brasil,diretoria-de-avaliacao,PDF,180541,0.17
1402,1340b2ea-d058-48c1-910a-7393dfd6d704,Link para os metadados dos dados de Docentes da Pós-Graduação,35eab2f8-5a64-4619-b3f1-63a2e6690cfa,[2013 a 2016] Docentes da Pós-Graduação Stricto Sensu no Brasil,Docentes da Pós-Graduação Stricto Sensu no Brasil,diretoria-de-avaliacao,HTML,0,0.00
1403,23a99b32-21c6-4787-9647-8296e1396553,Arquivo RDF Docentes da Pós-Graduação Stricto Sensu 2013 a 2016,35eab2f8-5a64-4619-b3f1-63a2e6690cfa,[2013 a 2016] Docentes da Pós-Graduação Stricto Sensu no Brasil,Docentes da Pós-Graduação Stricto Sensu no Brasil,diretoria-de-avaliacao,RDF,0,0.00


In [258]:
#Melhor mudar o nome da coluna 'Organização'
print(df_arquivos['Organização'].unique())

['bolsas-e-auxilios' 'acesso-ao-portal-de-periodicos'
 'diretoria-de-avaliacao' 'orcamento-e-financas-capes']


In [259]:

# Renomeando os valores da coluna Organização
mapeamento_org = {
    'diretoria-de-avaliacao': 'Avaliação da Pós-Graduação Stricto Sensu',
    'bolsas-e-auxilios': 'Bolsas e Auxílios',
    'orcamento-e-financas-capes': 'Orçamento e Finanças CAPES',
    'acesso-ao-portal-de-periodicos': 'Acessos ao Portal de Periódicos da CAPES'
}

# Substitui os valores antigos pelos novos
df_arquivos['Organização'] = df_arquivos['Organização'].replace(mapeamento_org)

print(df_arquivos['Organização'].unique())

['Bolsas e Auxílios' 'Acessos ao Portal de Periódicos da CAPES'
 'Avaliação da Pós-Graduação Stricto Sensu' 'Orçamento e Finanças CAPES']


In [260]:
#Salvando tabela com todos os arquivos
df_arquivos.to_csv(csv_arquivos_allformats_path, index=False)

In [273]:
#Mantendo apenas arquivos de formato CSV

df_csv = df_arquivos[df_arquivos['Formato'] == 'CSV']

print(df_csv.head(10))

                            Arquivo (ID)  \
0   48fe1036-db0d-40ce-a964-983407a5f105   
2   5452f35a-7ebb-4aae-8cc5-0f85df99d35d   
6   6ea46afe-485a-4f37-9752-6d5b7a1bbc04   
8   f810f11b-6c2b-44fa-9391-7d4752f8f054   
13  91520d26-e02a-4c8b-8bcc-e70b923dd7cf   
15  57457d2d-e69e-44b2-820a-5c4e47a8aef1   
17  b66a538c-bf1c-4008-88a3-ee59b5d934cc   
19  f02a5bbd-1654-4cd9-a6bf-d703e7d02a8b   
21  98a3cdb4-d925-4901-8fe7-186733f499d0   
23  39319ada-904c-4731-b53e-5b4c7abff10f   

                                                              Arquivo (Nome)  \
0   PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2013A2024-2025-06-01   
2        PARTICIPANTES-PROG-FORMACAO-PROFESSORES-EB-EXTERIOR-2025-2026-06-01   
6                       BR-CAPES-RECURSOS-INVESTIDOS-EB-2014A2024-2025-08-01   
8                            BR-CAPES-RECURSOS-INVESTIDOS-EB-2025-2026-06-01   
13                       PROJETOS-FORMACAO-INICIAL-PROF-2018a2025-2026-06-01   
15                            BR-CA

In [274]:
print(df_csv['Formato'].unique())

['CSV']


In [262]:
#Salvando tabela com todos os arquivos csv
df_csv.to_csv(csv_arquivos_onlycsv_path, index=False)

In [270]:
#Agrupamento pela coluna 'Grupo' e 'Organização'
df_grupo = df_csv.groupby(['Grupo', 'Organização']).agg(
    # Conta quantos IDs ÚNICOS de datasets existem neste grupo
    Quantidade_Datasets=('Dataset (ID)', 'nunique'),

    # CONTAGEM DE ARQUIVOS: Conta o número de linhas/arquivos desta organização
    Quantidade_Arquivos=('Arquivo (ID)', 'count'),

    
    # Soma os tamanhos de todos os arquivos deste grupo
    Tamanho_Total_Bytes=('Tamanho (Bytes)', 'sum'),
    Tamanho_Total_MB=('Tamanho (MB)', 'sum')
).reset_index()

#Calcula a porcentagem do tamanho total
tamanho_geral = df_grupo['Tamanho_Total_MB'].sum()

if tamanho_geral > 0:
    df_grupo['Porcentagem_%'] = (df_grupo['Tamanho_Total_MB'] / tamanho_geral) * 100
else:
    df_grupo['Porcentagem_%'] = 0

# Arredonda os valores numéricos para 2 casas decimais
df_grupo['Tamanho_Total_MB'] = df_grupo['Tamanho_Total_MB'].round(2)
df_grupo['Porcentagem_%'] = df_grupo['Porcentagem_%'].round(2)

# 4. Ordena baseado na organizaçao e tamanho total

#Ordenando a organização
ordem_organizacao = [
    'Avaliação da Pós-Graduação Stricto Sensu',
    'Bolsas e Auxílios', 
    'Orçamento e Finanças CAPES', 
    'Acessos ao Portal de Periódicos da CAPES',
]

# Transformando a coluna em "Categoria" e aplica a sua regra de ordem
df_grupo['Organização'] = pd.Categorical(
    df_grupo['Organização'], 
    categories=ordem_organizacao, 
    ordered=True
)

df_grupo = df_grupo.sort_values(by=['Organização', 'Tamanho_Total_MB'], ascending=[True, False]).reset_index(drop=True)

# Exibe o resultado final
df_grupo

,Grupo,Organização,Quantidade_Datasets,Quantidade_Arquivos,Tamanho_Total_Bytes,Tamanho_Total_MB,Porcentagem_%
0,Catálogo de Teses e Dissertações - Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,38,3800888539,"3,624.81",23.34
1,Produção Intelectual dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,124,2660077576,"2,536.83",16.33
2,Autor da Produção Intelectual de Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,94,2465244522,"2,351.04",15.14
3,Detalhes da Produção Intelectual dos Programas da Pós-Graduação Strictu Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,9,82,1829533767,"1,744.79",11.23
4,Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,14,1436166903,"1,369.63",8.82
5,Discentes da Pós-Graduação Stricto Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,21,1406554303,"1,341.39",8.64
6,Membros de Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,21,581854785,554.90,3.57
7,Docentes da Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,13,365446167,348.52,2.24
8,Financiadores de Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,12,166029723,158.34,1.02
9,Cursos da Pós-Graduação Stricto Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,12,9841252,9.38,0.06


In [264]:
#Salvando tabela com todos os grupos
df_grupo.to_csv(csv_grupos_onlycsv_posgrad_path, index=False)

In [271]:
#Tabela apenas com os grupos da Avaliação da Pós-Graduação

# 1. Filtra o DataFrame mantendo apenas a organização desejada
nome_organizacao = 'Avaliação da Pós-Graduação Stricto Sensu'
df_avaliacao = df_grupo[df_grupo['Organização'] == nome_organizacao].copy()

# 2. Calcula a nova soma total de MB APENAS para esta organização
tamanho_total_avaliacao = df_avaliacao['Tamanho_Total_MB'].sum()

# 3. Recalcula a porcentagem dentro do novo contexto
if tamanho_total_avaliacao > 0:
    df_avaliacao['Porcentagem_%'] = (df_avaliacao['Tamanho_Total_MB'] / tamanho_total_avaliacao) * 100
else:
    df_avaliacao['Porcentagem_%'] = 0

# 4. Arredonda a porcentagem recalculada
df_avaliacao['Porcentagem_%'] = df_avaliacao['Porcentagem_%'].round(2)

# 5. Ordena para que os maiores grupos fiquem no topo
df_avaliacao = df_avaliacao.sort_values(by='Tamanho_Total_MB', ascending=False).reset_index(drop=True)

# Exibe o resultado final
df_avaliacao

,Grupo,Organização,Quantidade_Datasets,Quantidade_Arquivos,Tamanho_Total_Bytes,Tamanho_Total_MB,Porcentagem_%
0,Catálogo de Teses e Dissertações - Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,38,3800888539,"3,624.81",25.81
1,Produção Intelectual dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,124,2660077576,"2,536.83",18.06
2,Autor da Produção Intelectual de Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,94,2465244522,"2,351.04",16.74
3,Detalhes da Produção Intelectual dos Programas da Pós-Graduação Strictu Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,9,82,1829533767,"1,744.79",12.42
4,Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,14,1436166903,"1,369.63",9.75
5,Discentes da Pós-Graduação Stricto Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,21,1406554303,"1,341.39",9.55
6,Membros de Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,21,581854785,554.90,3.95
7,Docentes da Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,4,13,365446167,348.52,2.48
8,Financiadores de Projetos dos Programas de Pós-Graduação Stricto Sensu no Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,12,166029723,158.34,1.13
9,Cursos da Pós-Graduação Stricto Sensu do Brasil,Avaliação da Pós-Graduação Stricto Sensu,3,12,9841252,9.38,0.07


In [266]:
#Salvando tabela com todos os grupos da avaliação
df_avaliacao.to_csv(csv_grupos_onlycsv_posgrad_path, index=False)

In [268]:
# 1. Agrupa apenas pela coluna 'Organização' a partir da tabela base (df_csv)
df_org = df_csv.groupby('Organização').agg(
    # Conta os IDs únicos de datasets pertencentes a essa organização
    Quantidade_Datasets=('Dataset (ID)', 'nunique'),

    # CONTAGEM DE ARQUIVOS: Conta o número de linhas/arquivos desta organização
    Quantidade_Arquivos=('Arquivo (ID)', 'count'),
    
    # Soma os tamanhos reais de todos os arquivos
    Tamanho_Total_Bytes=('Tamanho (Bytes)', 'sum'),
    Tamanho_Total_MB=('Tamanho (MB)', 'sum')
).reset_index()

# 2. Renomeia os slugs para os nomes reais (legíveis)
mapeamento_org = {
    'diretoria-de-avaliacao': 'Diretoria de Avaliação',
    'bolsas-e-auxilios': 'Bolsas e Auxílios',
    'orcamento-e-financas-capes': 'Orçamento e Finanças CAPES',
    'acesso-ao-portal-de-periodicos': 'Portal de Periódicos'
}
df_org['Organização'] = df_org['Organização'].replace(mapeamento_org)

# 3. Recalcula a porcentagem baseada no novo agrupamento
tamanho_geral_org = df_org['Tamanho_Total_MB'].sum()

if tamanho_geral_org > 0:
    df_org['Porcentagem_%'] = (df_org['Tamanho_Total_MB'] / tamanho_geral_org) * 100
else:
    df_org['Porcentagem_%'] = 0

# 4. Arredonda para 2 casas decimais
df_org['Tamanho_Total_MB'] = df_org['Tamanho_Total_MB'].round(2)
df_org['Porcentagem_%'] = df_org['Porcentagem_%'].round(2)

# 5. Ordena do maior para o menor tamanho em MB
df_org = df_org.sort_values(by='Tamanho_Total_MB', ascending=False).reset_index(drop=True)

# Exibe o resultado
df_org

,Organização,Quantidade_Datasets,Quantidade_Arquivos,Tamanho_Total_Bytes,Tamanho_Total_MB,Porcentagem_%
0,Avaliação da Pós-Graduação Stricto Sensu,45,443,14728817744,"14,046.48",90.43
1,Bolsas e Auxílios,32,104,1532653531,"1,461.64",9.41
2,Acessos ao Portal de Periódicos da CAPES,6,17,13443652,12.83,0.08
3,Orçamento e Finanças CAPES,10,23,12008678,11.46,0.07


In [241]:
#Salvando tabela com todas as Organizações
df_org.to_csv(csv_org_onlycsv_path, index=False)